In [1]:
from qiskit_qaoa.utils.routing_utils import greedy_parity_network, greedy_gaussian_elimination
from qiskit_qaoa.utils.swap_strategy import ExtendedSwapStrategy

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
import numpy as np

In [2]:
ess = ExtendedSwapStrategy.from_grid(2,3)
N = ess._num_vertices

qc = QuantumCircuit(N)

ops = (
    PauliEvolutionGate(SparsePauliOp('I'), time=1),
    PauliEvolutionGate(SparsePauliOp('Z'), time=2) ,
    PauliEvolutionGate(SparsePauliOp('ZZ'), time=3), 
    PauliEvolutionGate(SparsePauliOp('ZZZ'), time=4), 
    PauliEvolutionGate(SparsePauliOp('ZZZZ'), time=5)
)
current_layer = {
    (0,1): ops[2],
    (0,1,2,3): ops[4],
    (2,4,5): ops[3],
    (1,): ops[1]
}

In [3]:
ops[2].time

3.0

In [4]:
vector_inputs = [[int(z in x) for z in range(N)] for x in current_layer.keys()]
vector_inputs

[[1, 1, 0, 0, 0, 0],
 [1, 1, 1, 1, 0, 0],
 [0, 0, 1, 0, 1, 1],
 [0, 1, 0, 0, 0, 0]]

In [5]:
vector_current_layer = {tuple([int(z in x) for z in range(N)]): val for x, val in current_layer.items()}

In [6]:
A = greedy_parity_network(vector_current_layer, qc)

In [7]:
qc.draw(fold=-1)

q_0: ───────────■───────────────────────────────
     ┌───────┐┌─┴─┐┌───────┐                    
q_1: ┤ Rz(4) ├┤ X ├┤ Rz(6) ├──■─────────────────
     └───────┘└───┘└───────┘┌─┴─┐               
q_2: ────■──────────────────┤ X ├──■────────────
         │                  └───┘┌─┴─┐┌────────┐
q_3: ────┼───────────────────────┤ X ├┤ Rz(10) ├
       ┌─┴─┐                     └───┘└────────┘
q_4: ──┤ X ├────■───────────────────────────────
       └───┘  ┌─┴─┐┌───────┐                    
q_5: ─────────┤ X ├┤ Rz(8) ├────────────────────
              └───┘└───────┘

In [8]:
A

array([[1., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0.],
       [1., 1., 1., 0., 0., 0.],
       [1., 1., 1., 1., 0., 0.],
       [0., 0., 1., 0., 1., 0.],
       [0., 0., 1., 0., 1., 1.]])

In [9]:
greedy_gaussian_elimination(A, qc)

In [10]:
qc.draw(fold=-1)

q_0: ───────────■───────────────────────────────────────────■──
     ┌───────┐┌─┴─┐┌───────┐                              ┌─┴─┐
q_1: ┤ Rz(4) ├┤ X ├┤ Rz(6) ├──■────────────────────────■──┤ X ├
     └───────┘└───┘└───────┘┌─┴─┐                    ┌─┴─┐└───┘
q_2: ────■──────────────────┤ X ├──■──────────────■──┤ X ├──■──
         │                  └───┘┌─┴─┐┌────────┐┌─┴─┐└───┘  │  
q_3: ────┼───────────────────────┤ X ├┤ Rz(10) ├┤ X ├───────┼──
       ┌─┴─┐                     └───┘└────────┘└───┘     ┌─┴─┐
q_4: ──┤ X ├────■─────────────■───────────────────────────┤ X ├
       └───┘  ┌─┴─┐┌───────┐┌─┴─┐                         └───┘
q_5: ─────────┤ X ├┤ Rz(8) ├┤ X ├──────────────────────────────
              └───┘└───────┘└───┘

In [11]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_new import CommutingGateRouterNew
from qiskit.circuit.library import CXGate

In [12]:
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph

filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N2_W2.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
# ess = ExtendedSwapStrategy.from_grid(n, T)
ess = ExtendedSwapStrategy.from_all_to_all(n * T)
num_physical_qubits = ess._num_vertices
max_layers = 0

pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterNew(
            ess,
            max_layers=0,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

Keeping constraints at times: [0]
{}
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
Transpiling un-implemented gates


In [13]:
tqc.draw(fold=-1)

┌───────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         
q_0 -> 0 ┤ Rz(8.875) ├────────────■────────────────────────────■──────────────────■────────■────────────────────────────────■────────────────────────────────────■──────────────────────■──────────────────────────────■─────────────────────■────────────────────────────────────■────────────────────────────────────■────────────────────────────────────■─────────────────────────────────────────■─────────────────■────────────────────────────────────────■────────■────────■──────────────■────────────
         └───────────┘            │                            │                  │        │                                │                                    │                      │                              │                     │                                    │                                    │                                    │                                         │                 │                                        │        │        │              │            
q_1 -> 1 ─────────────────────────┼──────────────────■─────────┼──────────────────┼────────┼────────────────────────────────┼────────────────────────────────────┼──────────────────────┼──────────────────────────────┼─────────────────────┼────────────────────────────────────┼────────────────────────────────────┼────────────────────────────────────┼─────────────────────────────────────────┼─────────────────┼────────────────────────────────────────┼────────┼────────┼────■─────────┼─────────■──
                                  │                ┌─┴─┐       │  ┌───────────┐   │      ┌─┴─┐     ┌───────────┐            │                                    │                      │                              │                     │                                    │                                    │                                    │                                         │                 │                                        │      ┌─┴─┐      │  ┌─┴─┐       │         │  
q_2 -> 2 ────────────────■────────┼─────────■──────┤ X ├───────┼──┤ Rz(0.625) ├───┼──────┤ X ├─────┤ Rz(0.625) ├────────────┼─────────────■──────────────────────┼────────■─────────────┼─────────────────────■────────┼─────────────────────┼────────────────────────────────────┼──────────────────■─────────────────┼────────────────────────────────────┼──────────────────■────────■─────────────┼────────■────────┼──────────────────────■─────────────────┼──────┤ X ├──────┼──┤ X ├──■────┼─────────┼──
         ┌────────────┐  │      ┌─┴─┐       │  ┌───┴───┴────┐  │  └───────────┘   │      └───┘     └───────────┘            │           ┌─┴─┐     ┌───────────┐  │        │           ┌─┴─┐    ┌───────────┐  │        │                     │                                    │                  │                 │                                    │                  │      ┌─┴─┐           │      ┌─┴─┐      │                    ┌─┴─┐               │      └───┘      │  └───┘  │    │         │  
q_3 -> 3 ┤ Rz(-1.125) ├──┼──────┤ X ├───────┼──┤ Rz(-0.125) ├──┼──────────────────┼────────■───────────────────────■────────┼───────────┤ X ├─────┤ Rz(0.625) ├──┼────────┼───────────┤ X ├────┤ Rz(0.625) ├──┼────────┼─────────────────────┼────────────────────────────────────┼──────────────────┼─────────────────┼─────────────────■──────────────────┼──────────────────┼──────┤ X ├──────■────┼──────┤ X ├──────┼─────────────────■──┤ X ├───────────────┼─────────────────┼─────────┼────┼────■────┼──
         └────────────┘┌─┴─┐┌───┴───┴────